In [7]:
import os
import itertools
import multiprocessing as mp
import shutil

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

import numpy as np
import pandas as pd
import supervision as sv

from trackers import ByteTrackTracker
from trackers.eval import evaluate_mot_sequences


DATA_ROOT = "."
SPORTSMOT_DET_ROOT = os.path.join(DATA_ROOT, "sportsmot_yolox_dets")
SPORTSMOT_GT_ROOT = os.path.join(DATA_ROOT, "TrackEval", "data", "gt", "sportsmot")

MIN_BOX_AREA = 10
VERTICAL_RATIO_THRESH = 1.6


def get_yolox_detections(frame_id, det_list):
    dets_per_frame = [d for d in det_list if d.split(",")[0] == str(frame_id)]
    dets = []
    for line in dets_per_frame:
        det = line.split(",")
        x1, y1, x2, y2 = float(det[1]), float(det[2]), float(det[3]), float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame[frame_id]:
        det = line.split(",")
        x1, y1, x2, y2 = float(det[1]), float(det[2]), float(det[3]), float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets

In [11]:
def run_bytetrack_sportsmot_once(
    split: str,
    lost_track_buffer: int,
    track_activation_threshold: float,
    minimum_consecutive_frames: int,
    minimum_iou_threshold: float,
    high_conf_det_threshold: float,
    use_defaults: bool = False,
):
    assert split in {"val", "test"}

    if use_defaults:
        tracker = ByteTrackTracker()
    else:
        tracker = ByteTrackTracker(
            lost_track_buffer=lost_track_buffer,
            track_activation_threshold=track_activation_threshold,
            minimum_consecutive_frames=minimum_consecutive_frames,
            minimum_iou_threshold=minimum_iou_threshold,
            high_conf_det_threshold=high_conf_det_threshold,
        )

    det_root = os.path.join(SPORTSMOT_DET_ROOT, split)
    outputs_root = "BYTETRACK_outputs_sportsmot"
    os.makedirs(outputs_root, exist_ok=True)

    if use_defaults:
        save_dir = os.path.join(outputs_root, f"{split}_defaults")
    else:
        save_dir = os.path.join(
            outputs_root,
            f"{split}_L{lost_track_buffer}_A{track_activation_threshold}_"
            f"C{minimum_consecutive_frames}_I{minimum_iou_threshold}_H{high_conf_det_threshold}",
        )
    os.makedirs(save_dir, exist_ok=True)

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue
        tracker.reset()
        seq_name = seq.split(".")[0]
        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)
        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []
        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(xyxy=raw_dets[:, :4], confidence=raw_dets[:, 4])
            else:
                dets = sv.Detections.empty()
            dets = tracker.update(detections=dets)
            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue
                width = right - left
                height = bottom - top
                vertical = width / max(height, 1e-6) > VERTICAL_RATIO_THRESH
                if width * height > MIN_BOX_AREA and not vertical:
                    output_lines.append(
                        f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                    )
        save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

    HOTA = IDF1 = MOTA = None
    gt_dir = os.path.join(SPORTSMOT_GT_ROOT, split)
    try:
        result = evaluate_mot_sequences(gt_dir=gt_dir, tracker_dir=save_dir, metrics=["CLEAR", "HOTA", "Identity"])
        MOTA = result.to_dict()["aggregate"]["CLEAR"]["MOTA"]
        HOTA = result.to_dict()["aggregate"]["HOTA"]["HOTA"]
        IDF1 = result.to_dict()["aggregate"]["Identity"]["IDF1"]
        if use_defaults:
            print(f"split={split} | defaults -> HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}")
        else:
            print(f"split={split} | L:{lost_track_buffer}, A:{track_activation_threshold}, C:{minimum_consecutive_frames}, I:{minimum_iou_threshold}, H:{high_conf_det_threshold} -> HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f}")
    except Exception as e:
        print(f"split={split} | evaluation FAILED ({repr(e)}). Results in {save_dir}")

    return {
        "split": split,
        "lost_track_buffer": lost_track_buffer,
        "track_activation_threshold": track_activation_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "minimum_iou_threshold": minimum_iou_threshold,
        "high_conf_det_threshold": high_conf_det_threshold,
        "HOTA": HOTA, "IDF1": IDF1, "MOTA": MOTA,
        "output_dir": save_dir,
    }

In [12]:
# Run ByteTrack with default parameters (ByteTrackTracker()) on the definitive eval set (val)
res = run_bytetrack_sportsmot_once("test",  0.0, 0.0, 0, 0.0, 0.0, use_defaults=True)

submission_dir = res["output_dir"]
submission_zip_base = os.path.basename(submission_dir)
shutil.make_archive("bytetrack_"+submission_zip_base, "zip", submission_dir)

split=test | evaluation FAILED (FileNotFoundError('Ground truth directory not found: TrackEval/data/gt/sportsmot/test')). Results in BYTETRACK_outputs_sportsmot/test_defaults


'/Users/alexanderbodner/Documents/roboflow/trackers metrics/sportsmot/bytetrack_test_defaults.zip'

In [ ]:
lost_track_buffer_candidates = [10, 30, 60, 90]
track_activation_threshold_candidates = [0.2, 0.5, 0.7, 0.9]
minimum_consecutive_frames_candidates = [1, 3]
minimum_iou_threshold_candidates = [0.05, 0.1, 0.3]
high_conf_det_threshold_candidates = [0.5, 0.6, 0.7, 0.9]

all_candidate_lists = [
    lost_track_buffer_candidates,
    track_activation_threshold_candidates,
    minimum_consecutive_frames_candidates,
    minimum_iou_threshold_candidates,
    high_conf_det_threshold_candidates,
]
parameter_combinations = list(itertools.product(*all_candidate_lists))
print(f"Total ByteTrack parameter combinations: {len(parameter_combinations)}")

Total ByteTrack parameter combinations: 384


In [ ]:
def tune_bytetrack_on_val_parallel(parameter_combinations):
    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} ByteTrack combinations on val with {num_workers} workers")
    ctx = mp.get_context("fork")
    all_results = []
    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            L, A, C, I, H = comb
            futures.append(executor.submit(run_bytetrack_sportsmot_once, "val", L, A, C, I, H))
        for i, f in enumerate(futures):
            try:
                all_results.append(f.result())
                print(f"[{i+1}/{len(parameter_combinations)}] val combination finished.")
            except Exception as e:
                print(f"[{i+1}/{len(parameter_combinations)}] FAILED: {repr(e)}")
    df = pd.DataFrame(all_results)
    if not df.empty:
        df.to_csv("bytetrack_sportsmot_val_tuning.csv", index=False)
    return df

bytetrack_sportsmot_val_tuning_df = tune_bytetrack_on_val_parallel(parameter_combinations)

if not bytetrack_sportsmot_val_tuning_df.empty and "HOTA" in bytetrack_sportsmot_val_tuning_df.columns and not bytetrack_sportsmot_val_tuning_df["HOTA"].isna().all():
    best_idx = bytetrack_sportsmot_val_tuning_df["HOTA"].idxmax()
    best_row = bytetrack_sportsmot_val_tuning_df.loc[best_idx]
    best_params = dict(
        lost_track_buffer=int(best_row["lost_track_buffer"]),
        track_activation_threshold=float(best_row["track_activation_threshold"]),
        minimum_consecutive_frames=int(best_row["minimum_consecutive_frames"]),
        minimum_iou_threshold=float(best_row["minimum_iou_threshold"]),
        high_conf_det_threshold=float(best_row["high_conf_det_threshold"]),
    )
    print("Best validation row by HOTA:", best_row)
    print("Best params:", best_params)
else:
    print("No HOTA in results; set best_params manually below.")
    best_params = None

Running 384 ByteTrack combinations on val with 10 workers
split=val | L:10, F:30.0, A:0.2, C:1, I:0.1, H:0.6 -> HOTA: 0.800, IDF1: 0.794, MOTA: 0.982
split=val | L:10, F:30.0, A:0.2, C:1, I:0.1, H:0.5 -> HOTA: 0.798, IDF1: 0.791, MOTA: 0.982
split=val | L:10, F:30.0, A:0.2, C:1, I:0.05, H:0.6 -> HOTA: 0.801, IDF1: 0.795, MOTA: 0.982
split=val | L:10, F:30.0, A:0.2, C:1, I:0.1, H:0.7 -> HOTA: 0.804, IDF1: 0.800, MOTA: 0.982split=val | L:10, F:30.0, A:0.2, C:1, I:0.3, H:0.6 -> HOTA: 0.788, IDF1: 0.777, MOTA: 0.977

split=val | L:10, F:30.0, A:0.2, C:1, I:0.3, H:0.5 -> HOTA: 0.787, IDF1: 0.776, MOTA: 0.977
split=val | L:10, F:30.0, A:0.2, C:1, I:0.05, H:0.5 -> HOTA: 0.799, IDF1: 0.792, MOTA: 0.982
split=val | L:10, F:30.0, A:0.2, C:1, I:0.05, H:0.9 -> HOTA: 0.801, IDF1: 0.797, MOTA: 0.979
[1/384] val combination finished.
[2/384] val combination finished.
split=val | L:10, F:30.0, A:0.2, C:1, I:0.1, H:0.9 -> HOTA: 0.798, IDF1: 0.793, MOTA: 0.978
split=val | L:10, F:30.0, A:0.2, C:1, I:0.0

## Final test run and submission zip

In [ ]:
if best_params is None:
    best_params = dict(lost_track_buffer=30, track_activation_threshold=0.5, minimum_consecutive_frames=3, minimum_iou_threshold=0.3, high_conf_det_threshold=0.6)

print("Running ByteTrack on SportsMOT test with:", best_params)
test_result = run_bytetrack_sportsmot_once("test", **best_params)

submission_dir = test_result["output_dir"]
submission_zip_base = os.path.basename(submission_dir)
shutil.make_archive("bytetrack_"+submission_zip_base, "zip", submission_dir)
print("Created submission archive:", submission_zip_base + ".zip")

Running ByteTrack on SportsMOT test with: {'lost_track_buffer': 10, 'frame_rate': 30.0, 'track_activation_threshold': 0.9, 'minimum_consecutive_frames': 1, 'minimum_iou_threshold': 0.05, 'high_conf_det_threshold': 0.7}
split=test | evaluation FAILED (FileNotFoundError('Ground truth directory not found: TrackEval/data/gt/sportsmot/test')). Results in BYTETRACK_outputs_sportsmot/test_L10_F30.0_A0.9_C1_I0.05_H0.7
Created submission archive: test_L10_F30.0_A0.9_C1_I0.05_H0.7.zip


In [14]:
import pandas as pd

# Load validation tuning results (must include HOTA/IDF1/MOTA columns)
df = pd.read_csv("bytetrack_sportsmot_val_tuning_develop06_03.csv")

if "HOTA" not in df.columns or df["HOTA"].isna().all():
    raise ValueError(
        "No HOTA values found in 'ocsort_sportsmot_val_tuning.csv'. "
        "Make sure GT is available under SPORTSMOT_GT_ROOT and rerun the tuning cell."
    )

# Row with best HOTA on validation
best_idx = df["HOTA"].idxmax()
best = df.loc[best_idx]

best

split                                                                       val
lost_track_buffer                                                            10
frame_rate                                                                 30.0
track_activation_threshold                                                  0.9
minimum_consecutive_frames                                                    1
minimum_iou_threshold                                                      0.05
high_conf_det_threshold                                                     0.7
HOTA                                                                    0.80521
IDF1                                                                   0.802999
MOTA                                                                   0.979734
output_dir                    BYTETRACK_outputs_sportsmot/val_L10_F30.0_A0.9...
Name: 74, dtype: object